# BD Replica CRM — Command Center Notebook

Visión ejecutiva + técnica de `bd_replica_crm` desde VS Code.

**Objetivos**
- Ver si PostgreSQL local está vivo y qué tan fresco está.
- Inventariar esquemas, tablas, vistas y tamaños.
- Detectar tablas grandes, vacías o potencialmente obsoletas.
- Encontrar columnas temporales y estimar frescura.
- Revisar calidad básica: nulos, PKs y duplicados.
- Exponer objetos de observabilidad / Decision Intelligence si existen.
- Descubrir automáticamente objetos CRM/comerciales.
- Mantener todo en modo **solo lectura**.


## 0. Cómo ejecutarlo en VS Code

1. Coloca este archivo en `bd_replica_crm/notebooks/`.
2. Abre la raíz de `bd_replica_crm` en VS Code.
3. Selecciona el entorno Python del proyecto.
4. Si aún no instalaste el paquete local: `pip install -e .`
5. Asegúrate de tener `.env` configurado.
6. Ejecuta **Run All**.

> Usa `load_settings()` y `connect_postgres()` del propio repo. No guarda credenciales en el notebook.


In [ ]:
from __future__ import annotations

import sys
import time
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "pyproject.toml").exists() else cwd.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError(
        "No encuentro pyproject.toml. Abre este notebook dentro del repo bd_replica_crm."
    )

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from replica_cygnus.settings import load_settings
from replica_cygnus.connections import connect_postgres

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)

settings = load_settings(PROJECT_ROOT)

print("Repo:", PROJECT_ROOT)
print(
    "PostgreSQL:",
    f"{settings.postgres.host}:{settings.postgres.port}/{settings.postgres.database}",
)
print("Notebook iniciado:", datetime.now().astimezone().isoformat(timespec="seconds"))


In [ ]:
conn = connect_postgres(settings)

def df(sql: str, params=None) -> pd.DataFrame:
    t0 = time.perf_counter()
    out = pd.read_sql_query(sql, conn, params=params)
    out.attrs["elapsed_s"] = time.perf_counter() - t0
    return out

server = df("""
SELECT
    current_database() AS database,
    current_user AS usuario,
    version() AS version,
    now() AS server_now,
    pg_postmaster_start_time() AS postgres_started_at
""")

server


## 1. KPI técnico inmediato


In [ ]:
kpi = df("""
WITH rel AS (
    SELECT
        n.nspname AS schema_name,
        c.relname AS object_name,
        c.relkind,
        pg_total_relation_size(c.oid) AS bytes
    FROM pg_class c
    JOIN pg_namespace n ON n.oid = c.relnamespace
    WHERE n.nspname NOT IN ('pg_catalog', 'information_schema')
      AND n.nspname NOT LIKE 'pg_toast%'
      AND c.relkind IN ('r','p','v','m')
)
SELECT
    COUNT(DISTINCT schema_name) AS schemas,
    COUNT(*) FILTER (WHERE relkind IN ('r','p')) AS tables,
    COUNT(*) FILTER (WHERE relkind = 'v') AS views,
    COUNT(*) FILTER (WHERE relkind = 'm') AS materialized_views,
    pg_size_pretty(SUM(bytes)) AS total_size
FROM rel
""")

kpi


## 2. Mapa completo del Data Warehouse local


In [ ]:
inventory = df("""
SELECT
    n.nspname AS schema_name,
    c.relname AS object_name,
    CASE c.relkind
        WHEN 'r' THEN 'table'
        WHEN 'p' THEN 'partitioned_table'
        WHEN 'v' THEN 'view'
        WHEN 'm' THEN 'materialized_view'
        ELSE c.relkind::text
    END AS object_type,
    COALESCE(s.n_live_tup, c.reltuples)::bigint AS approx_rows,
    pg_total_relation_size(c.oid) AS bytes,
    pg_size_pretty(pg_total_relation_size(c.oid)) AS total_size,
    s.last_analyze,
    s.last_autoanalyze,
    s.last_vacuum,
    s.last_autovacuum
FROM pg_class c
JOIN pg_namespace n ON n.oid = c.relnamespace
LEFT JOIN pg_stat_user_tables s ON s.relid = c.oid
WHERE n.nspname NOT IN ('pg_catalog', 'information_schema')
  AND n.nspname NOT LIKE 'pg_toast%'
  AND c.relkind IN ('r','p','v','m')
ORDER BY pg_total_relation_size(c.oid) DESC, n.nspname, c.relname
""")

inventory.head(30)


In [ ]:
schema_summary = (
    inventory.groupby("schema_name", as_index=False)
    .agg(
        objects=("object_name", "count"),
        approx_rows=("approx_rows", "sum"),
        bytes=("bytes", "sum"),
    )
    .sort_values("bytes", ascending=False)
)

schema_summary["size_mb"] = schema_summary["bytes"] / 1024**2
schema_summary


### Tablas más pesadas

Útil para responder rápido: **¿dónde está el volumen real y qué podría estar haciendo lento un pipeline?**


In [ ]:
largest = inventory.query("object_type in ['table','partitioned_table']").copy()
largest["size_mb"] = largest["bytes"] / 1024**2

largest[
    ["schema_name","object_name","approx_rows","size_mb","last_autoanalyze"]
].head(25)


## 3. Columnas, PKs e índices


In [ ]:
columns = df("""
SELECT
    table_schema AS schema_name,
    table_name,
    ordinal_position,
    column_name,
    data_type,
    is_nullable
FROM information_schema.columns
WHERE table_schema NOT IN ('pg_catalog','information_schema')
ORDER BY table_schema, table_name, ordinal_position
""")

columns.groupby("schema_name").agg(
    tables=("table_name","nunique"),
    columns=("column_name","count")
).sort_values("tables", ascending=False)


In [ ]:
pk = df("""
SELECT
    ns.nspname AS schema_name,
    tbl.relname AS table_name,
    con.conname AS constraint_name,
    string_agg(att.attname, ', ' ORDER BY u.ord) AS pk_columns
FROM pg_constraint con
JOIN pg_class tbl ON tbl.oid = con.conrelid
JOIN pg_namespace ns ON ns.oid = tbl.relnamespace
CROSS JOIN LATERAL unnest(con.conkey) WITH ORDINALITY AS u(attnum, ord)
JOIN pg_attribute att
  ON att.attrelid = tbl.oid
 AND att.attnum = u.attnum
WHERE con.contype = 'p'
  AND ns.nspname NOT IN ('pg_catalog','information_schema')
GROUP BY ns.nspname, tbl.relname, con.conname
ORDER BY ns.nspname, tbl.relname
""")

pk


In [ ]:
indexes = df("""
SELECT
    schemaname AS schema_name,
    tablename AS table_name,
    indexname,
    indexdef
FROM pg_indexes
WHERE schemaname NOT IN ('pg_catalog','information_schema')
ORDER BY schemaname, tablename, indexname
""")

print("Índices:", len(indexes))
indexes.head(30)


## 4. Radar de frescura

Busca automáticamente columnas cuyo nombre sugiera fecha/hora y estima la última observación disponible.


In [ ]:
DATE_HINTS = (
    "fecha", "date", "created", "updated", "timestamp",
    "inicio", "fin", "carga", "sync", "etl", "inserted", "modified"
)

date_candidates = columns[
    columns["data_type"].str.contains("date|timestamp", case=False, na=False)
    & columns["column_name"].str.lower().apply(
        lambda x: any(h in x for h in DATE_HINTS)
    )
].copy()

priority_words = [
    "updated", "modified", "carga", "sync", "etl",
    "fecha_actualizacion", "fecha_modificacion",
    "created", "fecha_creacion", "fecha"
]

def priority(col):
    c = col.lower()
    for i, word in enumerate(priority_words):
        if word in c:
            return i
    return 999

date_candidates["priority"] = date_candidates["column_name"].map(priority)
date_candidates = date_candidates.sort_values(
    ["schema_name","table_name","priority","ordinal_position"]
)

date_candidates.head(40)


In [ ]:
MAX_FRESHNESS_TABLES = 60

table_sizes = inventory[
    inventory["object_type"].isin(["table","partitioned_table"])
][["schema_name","object_name","approx_rows","bytes"]].rename(
    columns={"object_name":"table_name"}
)

targets = (
    date_candidates
    .drop_duplicates(["schema_name","table_name"])
    .merge(table_sizes, on=["schema_name","table_name"], how="left")
    .sort_values(["bytes","approx_rows"], ascending=False)
    .head(MAX_FRESHNESS_TABLES)
)

freshness_rows = []

for r in targets.itertuples():
    schema, table, col = r.schema_name, r.table_name, r.column_name

    qschema = '"' + schema.replace('"','""') + '"'
    qtable = '"' + table.replace('"','""') + '"'
    qcol = '"' + col.replace('"','""') + '"'

    try:
        x = df(f'SELECT MAX({qcol}) AS max_ts FROM {qschema}.{qtable}')
        max_ts = x.iloc[0,0]
        freshness_rows.append((schema, table, col, max_ts, None))
    except Exception as exc:
        conn.rollback()
        freshness_rows.append(
            (schema, table, col, None, str(exc)[:180])
        )

freshness = pd.DataFrame(
    freshness_rows,
    columns=["schema_name","table_name","date_column","max_ts","error"]
)

freshness["max_ts"] = pd.to_datetime(
    freshness["max_ts"], errors="coerce", utc=True
)

now_utc = pd.Timestamp.now(tz="UTC")
freshness["age_hours"] = (
    now_utc - freshness["max_ts"]
).dt.total_seconds() / 3600

freshness.sort_values("age_hours", na_position="last").head(40)


In [ ]:
def freshness_label(hours):
    if pd.isna(hours):
        return "SIN FECHA"
    if hours <= 2:
        return "OK <=2h"
    if hours <= 24:
        return "REVISAR <=24h"
    if hours <= 72:
        return "STALE 1-3d"
    return "STALE >3d"

freshness["status"] = freshness["age_hours"].map(freshness_label)
freshness["status"].value_counts(dropna=False)


## 5. Radar de anomalías estructurales


In [ ]:
tables_only = inventory[
    inventory["object_type"].isin(["table","partitioned_table"])
].copy()

pk_keys = set(zip(pk["schema_name"], pk["table_name"]))

tables_only["has_pk"] = [
    (s,t) in pk_keys
    for s,t in zip(tables_only["schema_name"], tables_only["object_name"])
]

tables_only["size_mb"] = tables_only["bytes"] / 1024**2

tables_only["hours_since_autoanalyze"] = (
    pd.Timestamp.now(tz="UTC")
    - pd.to_datetime(
        tables_only["last_autoanalyze"],
        errors="coerce",
        utc=True
    )
).dt.total_seconds() / 3600

anomaly_radar = tables_only.assign(
    flag_empty=lambda x: x["approx_rows"].fillna(0).eq(0),
    flag_no_pk=lambda x: ~x["has_pk"],
    flag_large=lambda x: x["size_mb"].gt(100),
    flag_analyze_old=lambda x: x["hours_since_autoanalyze"].gt(24*7),
)

anomaly_radar["risk_points"] = (
    anomaly_radar["flag_empty"].astype(int)
    + anomaly_radar["flag_no_pk"].astype(int)
    + anomaly_radar["flag_large"].astype(int)
    + anomaly_radar["flag_analyze_old"].astype(int)
)

anomaly_radar.sort_values(
    ["risk_points","size_mb"], ascending=False
)[[
    "schema_name","object_name","approx_rows",
    "size_mb","has_pk","last_autoanalyze","risk_points"
]].head(40)


## 6. Capacidades de plataforma / Decision Intelligence existentes


In [ ]:
keywords = [
    "observ", "control", "decision", "experiment",
    "model", "feature", "quality", "audit",
    "score", "lead", "risk"
]

platform_objects = inventory[
    inventory["schema_name"].str.lower().apply(
        lambda s: any(k in s for k in keywords)
    )
    | inventory["object_name"].str.lower().apply(
        lambda s: any(k in s for k in keywords)
    )
].copy()

platform_objects[[
    "schema_name","object_name","object_type","approx_rows","total_size"
]].sort_values(["schema_name","object_name"]).head(100)


## 7. Descubrimiento CRM/comercial


In [ ]:
commercial_terms = [
    "cliente", "clientes", "lead",
    "interaccion", "interacciones",
    "proforma", "proformas",
    "proceso", "procesos",
    "separacion", "venta", "minuta",
    "unidad", "unidades", "proyecto"
]

commercial_objects = inventory[
    inventory["object_name"].str.lower().apply(
        lambda s: any(term in s for term in commercial_terms)
    )
].copy()

commercial_objects[[
    "schema_name","object_name","object_type","approx_rows","total_size"
]].sort_values(
    ["schema_name","approx_rows"],
    ascending=[True,False]
).head(100)


## 8. Perfil rápido de una tabla

Modifica `TARGET_SCHEMA` y `TARGET_TABLE` para explorar cualquier tabla.


In [ ]:
_candidates = commercial_objects[
    commercial_objects["object_type"].isin(["table","partitioned_table"])
]

if _candidates.empty:
    _candidates = inventory[
        inventory["object_type"].isin(["table","partitioned_table"])
    ]

TARGET_SCHEMA = _candidates.iloc[0]["schema_name"]
TARGET_TABLE = _candidates.iloc[0]["object_name"]

print("TARGET:", f"{TARGET_SCHEMA}.{TARGET_TABLE}")


In [ ]:
def quote_ident(x: str) -> str:
    return '"' + x.replace('"','""') + '"'

def profile_table(schema: str, table: str, sample_rows: int = 10):
    qschema, qtable = quote_ident(schema), quote_ident(table)

    meta = columns[
        (columns["schema_name"] == schema)
        & (columns["table_name"] == table)
    ].copy()

    sample = df(
        f"SELECT * FROM {qschema}.{qtable} LIMIT {int(sample_rows)}"
    )

    count = df(
        f"SELECT COUNT(*) AS rows FROM {qschema}.{qtable}"
    ).iloc[0,0]

    print(f"{schema}.{table}")
    print(f"Filas exactas: {count:,}")
    print(f"Columnas: {len(meta):,}")

    display(
        meta[
            ["ordinal_position","column_name","data_type","is_nullable"]
        ]
    )
    display(sample)

    return meta, sample, count

target_meta, target_sample, target_count = profile_table(
    TARGET_SCHEMA,
    TARGET_TABLE
)


## 9. Calidad de una tabla: nulos + cardinalidad


In [ ]:
SAMPLE_LIMIT = 50_000

def sample_quality(schema: str, table: str, limit: int = SAMPLE_LIMIT):
    qschema, qtable = quote_ident(schema), quote_ident(table)

    sample = df(
        f"SELECT * FROM {qschema}.{qtable} LIMIT {int(limit)}"
    )

    rows = []
    n = len(sample)

    for c in sample.columns:
        s = sample[c]

        rows.append({
            "column": c,
            "dtype": str(s.dtype),
            "sample_rows": n,
            "null_pct": float(s.isna().mean() * 100)
                if n else np.nan,
            "nunique": int(s.nunique(dropna=True))
                if n else 0,
            "unique_pct": float(
                s.nunique(dropna=True) / n * 100
            ) if n else np.nan,
        })

    return pd.DataFrame(rows).sort_values(
        ["null_pct","unique_pct"],
        ascending=[False,False]
    )

quality = sample_quality(TARGET_SCHEMA, TARGET_TABLE)
quality.head(50)


## 10. Duplicados potenciales sobre la PK


In [ ]:
target_pk = pk[
    (pk["schema_name"] == TARGET_SCHEMA)
    & (pk["table_name"] == TARGET_TABLE)
]

if target_pk.empty:
    print("⚠️ La tabla seleccionada no tiene PK declarada.")
else:
    pk_cols = [
        x.strip()
        for x in target_pk.iloc[0]["pk_columns"].split(",")
    ]

    select_cols = ", ".join(
        quote_ident(c) for c in pk_cols
    )

    qschema = quote_ident(TARGET_SCHEMA)
    qtable = quote_ident(TARGET_TABLE)

    dup = df(f"""
        SELECT {select_cols}, COUNT(*) AS n
        FROM {qschema}.{qtable}
        GROUP BY {select_cols}
        HAVING COUNT(*) > 1
        ORDER BY n DESC
        LIMIT 100
    """)

    print("PK:", pk_cols)
    print("Duplicados encontrados:", len(dup))
    display(dup)


## 11. Lectura ejecutiva automática


In [ ]:
largest_table = largest.iloc[0] if not largest.empty else None

fresh_ok = int(
    (freshness["age_hours"] <= 2).sum()
) if not freshness.empty else 0

fresh_total = int(
    freshness["age_hours"].notna().sum()
) if not freshness.empty else 0

no_pk = int((~tables_only["has_pk"]).sum())
empty = int(
    tables_only["approx_rows"].fillna(0).eq(0).sum()
)
total_tables = len(tables_only)

print("=== LECTURA EJECUTIVA ===")
print(
    f"• PostgreSQL responde y expone {total_tables:,} tablas "
    f"en {inventory['schema_name'].nunique():,} esquemas."
)

if largest_table is not None:
    print(
        f"• Mayor objeto físico: "
        f"{largest_table['schema_name']}.{largest_table['object_name']} "
        f"≈ {largest_table['approx_rows']:,.0f} filas / "
        f"{largest_table['size_mb']:.1f} MB."
    )

print(
    f"• Frescura detectable <=2h: "
    f"{fresh_ok}/{fresh_total} tablas inspeccionadas con fecha."
)

print(
    f"• Tablas sin PK declarada: "
    f"{no_pk}/{total_tables}."
)

print(
    f"• Tablas reportadas como vacías por estadísticas: "
    f"{empty}/{total_tables}."
)

print(
    f"• Objetos de plataforma/DI detectados: "
    f"{len(platform_objects):,}."
)

print(
    f"• Objetos CRM/comerciales detectados: "
    f"{len(commercial_objects):,}."
)

if fresh_total and fresh_ok / fresh_total < 0.5:
    print(
        "⚠️ Menos de la mitad de las tablas con fecha "
        "parecen frescas <=2h. Revisar si es esperado por dominio."
    )

if total_tables and no_pk > total_tables * 0.5:
    print(
        "⚠️ Más de la mitad de las tablas no declara PK. "
        "No siempre es error, pero complica unicidad y relaciones."
    )


## 12. Export opcional a `/reports`


In [ ]:
EXPORT = False

if EXPORT:
    out = PROJECT_ROOT / "reports" / "notebook_command_center"
    out.mkdir(parents=True, exist_ok=True)

    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    inventory.to_csv(
        out / f"inventory_{stamp}.csv",
        index=False
    )
    freshness.to_csv(
        out / f"freshness_{stamp}.csv",
        index=False
    )
    anomaly_radar.to_csv(
        out / f"anomaly_radar_{stamp}.csv",
        index=False
    )
    platform_objects.to_csv(
        out / f"platform_objects_{stamp}.csv",
        index=False
    )
    commercial_objects.to_csv(
        out / f"commercial_objects_{stamp}.csv",
        index=False
    )

    print("Exportado en:", out)
else:
    print(
        "EXPORT=False. Cambia a True "
        "si quieres persistir snapshots."
    )


## 13. Cierre limpio


In [ ]:
conn.close()
print("Conexión cerrada.")
